In [0]:
from pyspark.sql import functions as F

CATALOG, SCHEMA = "workspace", "default"
T_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_segments"
T_SILVER = f"{CATALOG}.{SCHEMA}.silver_telemetry"

bronze = spark.table(T_BRONZE)

In [0]:
display(bronze.groupBy("label", "anomaly").count().orderBy("label"))
display(bronze.select("timestamp").limit(5))

In [0]:
silver = (
    bronze.select(
        F.col("segment").cast("int").alias("segment"),
        F.col("channel"),
        F.to_timestamp("timestamp").alias("ts"),
        F.col("value").cast("double").alias("value"),
        F.col("anomaly").cast("int").alias("anomaly"),
        F.col("train").cast("int").alias("train"),
        F.col("sampling").cast("int").alias("sampling"),
    )
    .dropna(subset=["segment", "ts", "value"])
    .dropDuplicates(["segment", "ts"])
)

In [0]:
silver.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T_SILVER)
s = spark.table(T_SILVER)

n_bronze, n_silver = bronze.count(), s.count()
print(f"Bronze : {n_bronze:,} | Silver : {n_silver:,} | rejetées : {n_bronze - n_silver:,}")

display(s.agg(
    F.countDistinct("segment").alias("n_segments"),
    F.countDistinct("channel").alias("n_channels"),
    F.min("ts").alias("debut"),
    F.max("ts").alias("fin"),
))

In [0]:
incoherents = (
    s.groupBy("segment")
    .agg(F.countDistinct("channel").alias("n_ch"),
         F.countDistinct("anomaly").alias("n_lab"),
         F.countDistinct("train").alias("n_split"))
    .filter("n_ch > 1 OR n_lab > 1 OR n_split > 1")
)
print(f"Segments incohérents : {incoherents.count()}")